# Patient Visualization: Waveform + EHR Events

Interactive exploration of the canonical MIMIC-III output. Pick a patient, browse waveform segments, see EHR event timeline.

In [ ]:
import os, json, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

REPO_ROOT = Path(os.getcwd()).resolve()
if REPO_ROOT.name != "Physio_Data":
    REPO_ROOT = Path("/labs/hulab/mxwang/Physio_Data")

with open(REPO_ROOT / "indices" / "var_registry.json") as f:
    VAR_REGISTRY = json.load(f)

VAR_NAMES = {v["id"]: v["name"] for v in VAR_REGISTRY["variables"]}
VAR_CATS  = {v["id"]: v["category"] for v in VAR_REGISTRY["variables"]}

CAT_COLORS  = {"lab": "#2196F3", "vital": "#4CAF50", "action": "#FF9800", "score": "#F44336"}
CAT_MARKERS = {"lab": "o", "vital": "s", "action": "D", "score": "^"}

import yaml
with open(REPO_ROOT / "workzone" / "configs" / "server_paths.yaml") as f:
    PROCESSED_ROOT = yaml.safe_load(f)["mimic3"]["output_dir"]

print(f"Data root: {PROCESSED_ROOT}")
print(f"Patients: {len([d for d in os.listdir(PROCESSED_ROOT) if os.path.isdir(os.path.join(PROCESSED_ROOT, d)) and '_' in d])}")

## 1. Pick a patient

In [ ]:
# List patients sorted by EHR event count (pick one with rich data)
patients = []
for d in sorted(os.listdir(PROCESSED_ROOT)):
    meta_path = os.path.join(PROCESSED_ROOT, d, "meta.json")
    if not os.path.exists(meta_path):
        continue
    with open(meta_path) as f:
        m = json.load(f)
    patients.append({
        "dir": d,
        "n_seg": m.get("n_segments", 0),
        "n_ehr": m.get("n_ehr_events", 0),
        "hours": m.get("total_duration_hours", 0),
        "n_blocks": m.get("n_blocks", 1),
    })

patients.sort(key=lambda x: -x["n_ehr"])
print(f"Top 10 patients by EHR event count:")
print(f"{'dir':>20s}  {'segs':>6s}  {'EHR':>6s}  {'hours':>6s}  {'blocks':>6s}")
for p in patients[:10]:
    print(f"{p['dir']:>20s}  {p['n_seg']:6d}  {p['n_ehr']:6d}  {p['hours']:6.1f}  {p['n_blocks']:6d}")

In [ ]:
# ---- CHANGE THIS to explore different patients ----
PATIENT_DIR = os.path.join(PROCESSED_ROOT, patients[0]["dir"])
# PATIENT_DIR = os.path.join(PROCESSED_ROOT, "12345_67890")  # or paste a specific patient
# ----------------------------------------------------

meta = json.load(open(os.path.join(PATIENT_DIR, "meta.json")))
time_ms = np.load(os.path.join(PATIENT_DIR, "time_ms.npy"))
ehr = np.load(os.path.join(PATIENT_DIR, "ehr_events.npy"))
pleth = np.load(os.path.join(PATIENT_DIR, "PLETH40.npy"), mmap_mode='r')
ii = np.load(os.path.join(PATIENT_DIR, "II120.npy"), mmap_mode='r')

ref_ms = float(time_ms[0])
hours = (time_ms.astype(np.float64) - ref_ms) / 3600000.0
n_seg = len(time_ms)
stride_sec = meta.get("stride_sec", 25)

print(f"Patient: {meta.get('patient_id')}")
print(f"Segments: {n_seg},  Duration: {hours[-1]:.1f}h,  Blocks: {meta.get('n_blocks', '?')}")
print(f"PLETH40: {pleth.shape},  II120: {ii.shape}")
print(f"EHR events: {len(ehr)}")
print(f"Stride: {stride_sec}s,  Overlap: {meta.get('overlap_sec', '?')}s")
print(json.dumps(meta.get("per_channel", {}), indent=2))

## 2. EHR event summary

In [ ]:
# EHR event breakdown by variable
if len(ehr) > 0:
    print(f"{'var_id':>6s}  {'name':>20s}  {'cat':>7s}  {'count':>6s}  {'min':>8s}  {'max':>8s}  {'median':>8s}")
    print("-" * 75)
    for vid in sorted(set(int(v) for v in ehr['var_id'])):
        mask = ehr['var_id'] == vid
        vals = ehr['value'][mask]
        name = VAR_NAMES.get(vid, f"?{vid}")
        cat = VAR_CATS.get(vid, "?")
        print(f"{vid:6d}  {name:>20s}  {cat:>7s}  {len(vals):6d}  {vals.min():8.2f}  {vals.max():8.2f}  {np.median(vals):8.2f}")

## 3. EHR event timeline (full recording)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

if len(ehr) > 0:
    evt_hours = (ehr['time_ms'].astype(np.float64) - ref_ms) / 3600000.0
    for cat in ["lab", "vital", "action", "score"]:
        mask = np.array([VAR_CATS.get(int(v), "") == cat for v in ehr['var_id']])
        if not np.any(mask):
            continue
        ax.scatter(evt_hours[mask], ehr['var_id'][mask].astype(np.float32),
                   c=CAT_COLORS[cat], marker=CAT_MARKERS[cat],
                   s=15, alpha=0.6, linewidths=0, label=cat)

    unique_vids = sorted(set(int(v) for v in ehr['var_id']))
    if len(unique_vids) <= 40:
        ax.set_yticks(unique_vids)
        ax.set_yticklabels([f"{v} {VAR_NAMES.get(v, '?')}" for v in unique_vids], fontsize=8)

ax.set_xlabel("Hours from recording start", fontsize=11)
ax.set_ylabel("Variable", fontsize=11)
ax.set_title(f"EHR Events  |  {meta.get('patient_id')}  |  {len(ehr)} events", fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 4. Waveform detail (change `SEG` to browse)

In [ ]:
# ---- CHANGE THIS to browse different segments ----
SEG = n_seg // 2          # segment index (0 to n_seg-1)
# SEG = int(3.5 * 3600 / stride_sec)   # or jump to ~3.5 hours
# ---------------------------------------------------

seg_dur = meta.get("segment_duration_sec", 30)

fig, axes = plt.subplots(2, 1, figsize=(16, 5), sharex=True)

# PLETH
p_data = pleth[SEG].astype(np.float32)
p_time = np.linspace(0, seg_dur, len(p_data))
p_nan = np.isnan(p_data).mean() * 100
if p_nan < 100:
    axes[0].plot(p_time, p_data, color='#1565C0', linewidth=0.5)
axes[0].set_ylabel("PLETH40 (NU)")
axes[0].set_title(f"Segment {SEG}  |  t = {hours[SEG]:.2f}h  |  PLETH NaN: {p_nan:.0f}%", loc='left')

# II
i_data = ii[SEG].astype(np.float32)
i_time = np.linspace(0, seg_dur, len(i_data))
i_nan = np.isnan(i_data).mean() * 100
if i_nan < 100:
    axes[1].plot(i_time, i_data, color='#C62828', linewidth=0.3)
axes[1].set_ylabel("II120 (mV)")
lbl = f"II NaN: {i_nan:.0f}%"
if i_nan == 100:
    lbl += "  (ECG absent)"
axes[1].set_title(lbl, loc='left')
axes[1].set_xlabel("Time within segment (s)")

# Mark EHR events that fall in this segment
seg_events = ehr[ehr['seg_idx'] == SEG]
if len(seg_events) > 0:
    for ev in seg_events:
        offset_s = (float(ev['time_ms']) - float(time_ms[SEG])) / 1000
        offset_s = np.clip(offset_s, 0, seg_dur)
        name = VAR_NAMES.get(int(ev['var_id']), str(ev['var_id']))
        cat = VAR_CATS.get(int(ev['var_id']), "?")
        color = CAT_COLORS.get(cat, "gray")
        for ax in axes:
            ax.axvline(offset_s, color=color, alpha=0.5, linewidth=1, linestyle='--')
        axes[0].annotate(f"{name}={ev['value']:.1f}", xy=(offset_s, axes[0].get_ylim()[1]),
                         fontsize=7, rotation=30, color=color, ha='left', va='bottom')
    print(f"EHR events in segment {SEG}: {len(seg_events)}")
    for ev in seg_events:
        print(f"  {VAR_NAMES.get(int(ev['var_id']), '?'):>20s} = {ev['value']:.2f}")

plt.tight_layout()
plt.show()

## 5. Signal overview + gaps (full recording)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 5), sharex=True)

# Compute per-segment envelope (subsample for speed)
step = max(1, n_seg // 2000)
idxs = np.arange(0, n_seg, step)
seg_h = hours[idxs]

for ax, (ch_name, ch_arr, color) in zip(axes, [
    ("PLETH40", pleth, "#1565C0"), ("II120", ii, "#C62828")
]):
    means, stds, nans = [], [], []
    for i in idxs:
        seg = ch_arr[i].astype(np.float32)
        nf = np.isnan(seg).mean()
        nans.append(nf)
        if nf < 1.0:
            valid = seg[~np.isnan(seg)]
            means.append(np.mean(valid))
            stds.append(np.std(valid))
        else:
            means.append(np.nan); stds.append(np.nan)

    means, stds = np.array(means), np.array(stds)
    ok = ~np.isnan(means)
    if np.any(ok):
        ax.fill_between(seg_h[ok], means[ok]-stds[ok], means[ok]+stds[ok], alpha=0.3, color=color)
        ax.plot(seg_h[ok], means[ok], color=color, linewidth=0.5)
    ax.set_ylabel(ch_name)

    # NaN segments
    nan_arr = np.array(nans)
    all_nan = nan_arr >= 1.0
    if np.any(all_nan):
        for h in seg_h[all_nan]:
            ax.axvline(h, color='red', alpha=0.1, linewidth=0.5)

# Mark gaps (time_ms jumps)
if n_seg > 1:
    diffs = np.diff(time_ms)
    gaps = np.where(diffs > stride_sec * 1000 * 1.5)[0]
    for gi in gaps:
        for ax in axes:
            ax.axvline(hours[gi], color='red', alpha=0.6, linewidth=1.5, linestyle='--')
    print(f"Gaps: {len(gaps)}")

axes[1].set_xlabel("Hours from recording start")
axes[0].set_title(f"Signal envelope (mean +/- std)  |  red dashes = recording gaps", loc='left')
plt.tight_layout()
plt.show()

## 6. Single EHR variable over time (change `VAR_ID` to explore)

In [ ]:
# ---- CHANGE THIS to plot different variables ----
VAR_ID = 100   # HR (try: 0=K, 5=Cr, 6=Bili, 100=HR, 101=SpO2, 200=vasopressor, 203=FiO2)
# -------------------------------------------------

var_mask = ehr['var_id'] == VAR_ID
var_events = ehr[var_mask]
name = VAR_NAMES.get(VAR_ID, f"var_{VAR_ID}")

if len(var_events) > 0:
    evt_h = (var_events['time_ms'].astype(np.float64) - ref_ms) / 3600000.0
    vals = var_events['value']
    cat = VAR_CATS.get(VAR_ID, "?")

    fig, ax = plt.subplots(figsize=(14, 3.5))
    ax.plot(evt_h, vals, 'o-', color=CAT_COLORS.get(cat, "gray"), markersize=4, linewidth=1, alpha=0.7)
    ax.set_xlabel("Hours from recording start")
    ax.set_ylabel(f"{name} ({VAR_REGISTRY['variables'][[v['id'] for v in VAR_REGISTRY['variables']].index(VAR_ID)]['unit']})")
    ax.set_title(f"{name} (var_id={VAR_ID})  |  {len(var_events)} measurements", fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print(f"No events for {name} (var_id={VAR_ID})")